# Entrenamiento PKLot: clasificador libre/ocupado con MobileNetV2
Notebook pensado para correr en Google Colab. Entrena un clasificador binario
sobre patches del dataset PKLot y exporta el modelo final a Google Drive.

In [ ]:
# Celda 1: Instalacion/import de dependencias
!pip install -q tensorflow scikit-learn matplotlib seaborn

import os
import json
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

In [ ]:
# Celda 2: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Celda 3: Configuracion (editar estas variables antes de correr el notebook)
ZIP_PATH     = "/content/drive/MyDrive/PKLot.zip"
DATASET_PATH = "/content/PKLot_data"
OUTPUT_PATH  = "/content/drive/MyDrive/PKLot_output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

IMAGE_SIZE    = (96, 96)
BATCH_SIZE    = 16
EPOCHS        = 20
LEARNING_RATE = 1e-4
RANDOM_SEED   = 42
MAX_IMAGES    = 2000   # None para usar todas las imagenes del dataset

In [ ]:
# Celda 3.1: Descomprimir el dataset
# Copiar el .zip a Colab y descomprimirlo localmente es mucho mas rapido que
# leer cientos de miles de archivos chicos directo desde Drive por red.
import shutil
import zipfile

if not os.path.isdir(DATASET_PATH):
    local_zip_path = "/content/PKLot.zip"
    print("Copiando el zip de Drive a Colab...")
    shutil.copy(ZIP_PATH, local_zip_path)
    print("Descomprimiendo...")
    with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
        zip_ref.extractall(DATASET_PATH)
    print(f"Dataset descomprimido en: {DATASET_PATH}")
else:
    print(f"Ya existe {DATASET_PATH}, no se vuelve a descomprimir.")

## Exploracion de la estructura de carpetas
PKLot puede venir con estructuras distintas segun como se descargue
(por ejemplo `PKLot/PKLotSegmented/UFPR04/Cloudy/2012-09-12/Empty/...`).
Esta celda recorre `DATASET_PATH` y muestra el arbol de carpetas y cuantas
imagenes Empty/Occupied encuentra, para verificar el path ANTES de entrenar.

In [ ]:
# Celda 4: Explorar la estructura real del dataset
def print_tree(root, max_depth=6, _depth=0):
    if _depth > max_depth:
        return
    root = Path(root)
    entries = sorted([p for p in root.iterdir() if p.is_dir()])
    for entry in entries:
        print("  " * _depth + f"- {entry.name}/")
        print_tree(entry, max_depth=max_depth, _depth=_depth + 1)

print(f"Explorando: {DATASET_PATH}\n")
print_tree(DATASET_PATH)

# Buscar archivos de metadata (data.yaml, _annotations.csv, classes.txt, etc.)
print("\n--- Archivos de metadata ---")
meta_exts = {".yaml", ".yml", ".csv", ".txt", ".json"}
meta_files = [p for p in Path(DATASET_PATH).rglob("*") if p.suffix.lower() in meta_exts and p.is_file()]
for mf in meta_files[:10]:
    print(f"  {mf.relative_to(DATASET_PATH)}")
if not meta_files:
    print("  (ninguno encontrado)")

# Muestra 10 nombres de archivo del train para detectar patrones de clase
print("\n--- Primeros 10 archivos en train/ (ordenados) ---")
train_files = sorted(f.name for f in (Path(DATASET_PATH) / "train").iterdir() if f.is_file())
for name in train_files[:10]:
    print(f"  {name}")

# Buscar si "empty" u "occupied" aparece en algún nombre de archivo
import re
sample = train_files[:500]
has_empty  = any("empty"    in n.lower() for n in sample)
has_occ    = any("occupied" in n.lower() for n in sample)
has_free   = any("free"     in n.lower() for n in sample)
print(f"\n¿'empty' en nombres?    {has_empty}")
print(f"¿'occupied' en nombres? {has_occ}")
print(f"¿'free' en nombres?     {has_free}")

# Contar todas las imágenes
all_images = list(Path(DATASET_PATH).rglob("*.jpg")) + \
             list(Path(DATASET_PATH).rglob("*.jpeg")) + \
             list(Path(DATASET_PATH).rglob("*.png"))
print(f"\nTotal imagenes: {len(all_images)}")
assert len(all_images) > 0, f"No hay imagenes en {DATASET_PATH}."

## Carga del dataset y split train/val/test
Buscamos todas las imagenes bajo carpetas `Empty`/`Occupied` (sin importar
mayusculas) y armamos pares (ruta, label). Si detectamos subcarpetas con
formato de fecha (PKLot trae `Cloudy/2012-09-12/...`), agrupamos el split por
fecha para que todos los frames de un mismo dia queden en el mismo split y
evitar data leakage entre frames del mismo video. Si no hay esa estructura,
hacemos split aleatorio estratificado por clase.

In [ ]:
# Celda 5: Recolectar patches desde anotaciones COCO
import json as _json

def collect_image_label_pairs(dataset_path):
    dataset_path = Path(dataset_path)
    empty_kw    = {"space-empty", "empty", "free"}
    occupied_kw = {"space-occupied", "occupied", "busy"}
    img_exts    = {".jpg", ".jpeg", ".png"}
    pairs = []

    for split in ["train", "valid", "test"]:
        split_path = dataset_path / split
        if not split_path.exists():
            continue
        coco_path = split_path / "_annotations.coco.json"

        if coco_path.exists():
            with open(coco_path) as f:
                coco = _json.load(f)
            cat_label = {}
            for cat in coco["categories"]:
                name = cat["name"].lower()
                if any(kw in name for kw in empty_kw):
                    cat_label[cat["id"]] = 0
                elif any(kw in name for kw in occupied_kw):
                    cat_label[cat["id"]] = 1
            id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
            before = len(pairs)
            for ann in coco["annotations"]:
                label = cat_label.get(ann["category_id"])
                if label is None:
                    continue
                fname = id_to_file.get(ann["image_id"])
                if not fname:
                    continue
                pairs.append((str(split_path / fname), ann["bbox"], label, split))
            print(f"  {split}: {len(pairs) - before} patches (COCO)")
        else:
            import re
            date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
            for label_name, label in [("Empty",0),("empty",0),("Occupied",1),("occupied",1)]:
                for img_path in split_path.rglob(f"{label_name}/*"):
                    if img_path.is_file() and img_path.suffix.lower() in img_exts:
                        parent = img_path.parent.parent.name
                        date_key = parent if date_pattern.match(parent) else split
                        pairs.append((str(img_path), None, label, date_key))

    return pairs


pairs = collect_image_label_pairs(DATASET_PATH)
uses_coco = len(pairs) > 0 and pairs[0][1] is not None
print(f"\nTotal patches antes de subsampling: {len(pairs)}")
assert len(pairs) > 0, "No se encontraron pares. Revisa DATASET_PATH."

# Subsampling: limitar la cantidad de imagenes fuente usadas
if MAX_IMAGES is not None:
    from collections import defaultdict as _dd
    random.seed(RANDOM_SEED)

    # Agrupar anotaciones por (split, imagen fuente)
    by_split_image = _dd(list)
    for pair in pairs:
        by_split_image[(pair[3], pair[0])].append(pair)

    # Contar imagenes unicas por split
    split_totals = _dd(int)
    for (split, _) in by_split_image:
        split_totals[split] += 1
    total_imgs = sum(split_totals.values())

    # Samplear proporcionalmente en cada split
    sampled = []
    for split in ["train", "valid", "test"]:
        n_split  = split_totals[split]
        n_sample = max(1, round(MAX_IMAGES * n_split / total_imgs))
        img_keys = [k for k in by_split_image if k[0] == split]
        chosen   = random.sample(img_keys, min(n_sample, len(img_keys)))
        for key in chosen:
            sampled.extend(by_split_image[key])

    pairs = sampled
    unique_imgs = len({p[0] for p in pairs})
    print(f"Subsampling: {unique_imgs} imagenes fuente -> {len(pairs)} patches")

has_date_structure  = (not uses_coco) and all(
    s not in ("train", "valid", "test") for _, _, _, s in pairs
)
has_roboflow_splits = not has_date_structure

In [ ]:
# Celda 6: Split train/val/test
random.seed(RANDOM_SEED)

if has_roboflow_splits or uses_coco:
    # Dataset ya viene con splits de Roboflow: train / valid / test
    train_pairs = [(p, b, l) for p, b, l, s in pairs if s == "train"]
    val_pairs   = [(p, b, l) for p, b, l, s in pairs if s == "valid"]
    test_pairs  = [(p, b, l) for p, b, l, s in pairs if s == "test"]
    print("Usando splits pre-existentes (train/valid/test).")

elif has_date_structure:
    # Split por fecha para evitar data leakage entre frames del mismo dia
    dates = sorted(set(s for _, _, _, s in pairs))
    random.shuffle(dates)
    n = len(dates)
    train_dates = set(dates[:int(0.7 * n)])
    val_dates   = set(dates[int(0.7 * n):int(0.85 * n)])
    test_dates  = set(dates[int(0.85 * n):])
    train_pairs = [(p, b, l) for p, b, l, s in pairs if s in train_dates]
    val_pairs   = [(p, b, l) for p, b, l, s in pairs if s in val_dates]
    test_pairs  = [(p, b, l) for p, b, l, s in pairs if s in test_dates]

else:
    # Fallback: split aleatorio estratificado por clase
    all_paths  = [p for p, b, l, s in pairs]
    all_bboxes = [b for p, b, l, s in pairs]
    all_labels = [l for p, b, l, s in pairs]
    idxs = list(range(len(all_paths)))
    train_idx, rest_idx = train_test_split(
        idxs, test_size=0.3, stratify=all_labels, random_state=RANDOM_SEED
    )
    rest_labels = [all_labels[i] for i in rest_idx]
    val_idx, test_idx = train_test_split(
        rest_idx, test_size=0.5, stratify=rest_labels, random_state=RANDOM_SEED
    )
    train_pairs = [(all_paths[i], all_bboxes[i], all_labels[i]) for i in train_idx]
    val_pairs   = [(all_paths[i], all_bboxes[i], all_labels[i]) for i in val_idx]
    test_pairs  = [(all_paths[i], all_bboxes[i], all_labels[i]) for i in test_idx]


def count_by_label(split_pairs):
    counts = defaultdict(int)
    for _, _, label in split_pairs:
        counts[label] += 1
    return dict(counts)


print(f"Train: {len(train_pairs):>7} patches -> {count_by_label(train_pairs)}")
print(f"Val:   {len(val_pairs):>7} patches -> {count_by_label(val_pairs)}")
print(f"Test:  {len(test_pairs):>7} patches -> {count_by_label(test_pairs)}")

## Pipeline de datos con augmentation
Armamos `tf.data.Dataset` para cada split. El de train tiene augmentation
(flip horizontal, brillo, rotacion leve) ya que PKLot tiene variacion de
clima/luz; val y test no llevan augmentation.

In [ ]:
# Celda 7: Construccion de los tf.data.Dataset
_rotation_layer = tf.keras.layers.RandomRotation(0.04)


def load_and_crop(path, bbox, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.cast(image, tf.float32)

    if uses_coco:
        img_h = tf.cast(tf.shape(image)[0], tf.float32)
        img_w = tf.cast(tf.shape(image)[1], tf.float32)
        x = bbox[0] / img_w
        y = bbox[1] / img_h
        w = bbox[2] / img_w
        h = bbox[3] / img_h
        box = tf.reshape(tf.stack([
            tf.clip_by_value(y,     0.0, 1.0),
            tf.clip_by_value(x,     0.0, 1.0),
            tf.clip_by_value(y + h, 0.0, 1.0),
            tf.clip_by_value(x + w, 0.0, 1.0),
        ]), [1, 4])
        image = tf.image.crop_and_resize(
            tf.expand_dims(image, 0), box, [0], IMAGE_SIZE
        )[0] / 255.0
    else:
        image = tf.image.resize(image, IMAGE_SIZE) / 255.0

    return image, tf.cast(label, tf.float32)


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = _rotation_layer(tf.expand_dims(image, 0))[0]
    return image, label


def make_dataset(split_pairs, training):
    paths  = [p for p, b, l in split_pairs]
    bboxes = [[float(v) for v in b] if b is not None else [0., 0., 0., 0.]
              for p, b, l in split_pairs]
    labels = [l for p, b, l in split_pairs]
    ds = tf.data.Dataset.from_tensor_slices((paths, bboxes, labels))
    # Limitar paralelismo de carga para no tener N imagenes 640x640 en RAM simultaneamente
    ds = ds.map(load_and_crop, num_parallel_calls=4)
    if training:
        # Buffer pequeño: 2000 patches * 96x96x3xfloat16 ≈ 100 MB
        ds = ds.shuffle(buffer_size=2000, seed=RANDOM_SEED)
        ds = ds.map(augment, num_parallel_calls=4)
    ds = ds.batch(BATCH_SIZE).prefetch(2)
    return ds


train_ds = make_dataset(train_pairs, training=True)
val_ds   = make_dataset(val_pairs,   training=False)
test_ds  = make_dataset(test_pairs,  training=False)

## Modelo: transfer learning con MobileNetV2
Usamos MobileNetV2 preentrenado en ImageNet como extractor de caracteristicas
(congelado), con una cabeza de clasificacion binaria simple encima.

In [ ]:
# Celda 8: Definicion del modelo
# Mixed precision: computa en float16, acumula gradientes en float32.
# Reduce uso de VRAM ~2x sin impacto en accuracy.
tf.keras.mixed_precision.set_global_policy("mixed_float16")

base_model = MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    # dtype=float32 en la salida es necesario con mixed precision para estabilidad numerica
    layers.Dense(1, activation="sigmoid", dtype="float32"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
)

model.summary()

## Entrenamiento con callbacks
EarlyStopping evita sobreentrenar, ModelCheckpoint guarda el mejor modelo
visto durante el entrenamiento, y ReduceLROnPlateau baja el learning rate
cuando el val_loss se estanca.

In [ ]:
# Celda 9: Entrenamiento
checkpoint_path = os.path.join(OUTPUT_PATH, "best_checkpoint.h5")

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## Evaluacion sobre el set de test
Calculamos accuracy, precision, recall, F1 y la matriz de confusion sobre
imagenes que el modelo nunca vio durante el entrenamiento.

In [ ]:
# Celda 10: Evaluacion en test
y_true = []
y_pred = []

for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0).flatten()
    y_pred.extend((probabilities >= 0.5).astype(int).tolist())
    y_true.extend(labels.numpy().astype(int).tolist())

print(classification_report(y_true, y_pred, target_names=["Empty", "Occupied"]))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Empty", "Occupied"]).plot(cmap="Blues")
plt.title("Matriz de confusion - Test set")
plt.show()

## Curvas de entrenamiento
Graficamos loss y accuracy de train vs val por epoch para ver si hubo
overfitting o underfitting.

In [ ]:
# Celda 11: Graficos de loss y accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## Exportar el modelo final a Google Drive
Guardamos el modelo entrenado en `OUTPUT_PATH` en formato `.h5`, listo para
descargar y usar localmente con `inference/detect.py`.

In [ ]:
# Celda 12: Exportar modelo final
final_model_path = os.path.join(OUTPUT_PATH, "modelo_pklot.h5")
model.save(final_model_path)
print(f"Modelo guardado en: {final_model_path}")
print("Descargalo desde Google Drive y usalo con --model en inference/detect.py")